# Отчёт по домашнему заданию: обучение tool-using агента с GRPO

**Автор:** 

Этот ноутбук написан под локальную структуру проекта вида `hw4/...` и не требует распаковки архива.

В отчёте описаны:
- постановка задачи;
- устройство среды, инструментов и verifier;
- baseline-результаты;
- результаты обучения с помощью GRPO;
- интерпретация того, почему обученная политика деградировала;
- выводы и возможные улучшения.

## 1. Постановка задачи

В этой работе решается задача **обучения агента, который пользуется инструментами** для ответа на вопросы в специализированной среде.

По структуре проекта видно, что агент работает не в абстрактной числовой среде, а в задаче с:
- отдельной средой `env/`;
- агентной логикой `agent/`;
- verifier `verifier/`;
- обучением `training/`;
- baseline-оценкой и логами эпизодов.

Содержательно задача состоит в том, чтобы агент последовательно выполнять действия, обращаться к инструментам, собирать полезную информацию и в конце выдавать финальный ответ. Качество поведения оценивается не только по финальному успеху, но и по промежуточным характеристикам траектории: длине эпизода, числу tool calls, нарушениям policy rules и другим вспомогательным сигналам.

Таким образом, это задача **RL для tool-using агента**, где нужно не просто получить правильный ответ, а выучить устойчивую стратегию взаимодействия со средой.

## 2. Что было сделано

В рамках проекта были реализованы или использованы следующие компоненты:

1. **Среда** для многошагового взаимодействия агента с задачей.
2. **Набор инструментов**, которыми агент может пользоваться по ходу решения.
3. **Verifier**, который проверяет корректность финального ответа и вычисляет метрики эпизода.
4. **Reward function** для RL-обучения, в которой объединяются:
   - итоговый успех/неуспех;
   - shaping-сигналы за полезное или вредное поведение;
   - штрафы за лишние шаги, некорректные действия и нарушения правил.
5. **Baseline evaluation** — оценка исходной политики до обучения.
6. **Обучение с помощью GRPO**.
7. **Сравнение baseline и GRPO** на нескольких eval-бакетах сложности.

Основная цель эксперимента — проверить, улучшает ли GRPO поведение агента в такой среде, и если нет, то объяснить, почему это произошло.

In [ ]:
from pathlib import Path
import json
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

# Поиск корня проекта: либо текущая папка, либо ./hw4
candidates = [Path('.').resolve(), Path('./hw4').resolve()]
PROJECT_ROOT = None
for c in candidates:
    if (c / 'training').exists() and (c / 'graphics').exists():
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Не удалось найти корень проекта. Открой ноутбук из папки проекта или положи его рядом с каталогом hw4/.')

GRAPHICS_DIR = PROJECT_ROOT / 'graphics'
BASELINE_DIR = PROJECT_ROOT / 'logs_baseline'
TRAINING_DIR = PROJECT_ROOT / 'training'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('GRAPHICS_DIR =', GRAPHICS_DIR)
print('BASELINE_DIR =', BASELINE_DIR)

## 3. Структура проекта

Ниже автоматически выводится краткая сводка по ключевым частям проекта. Это помогает зафиксировать, что работа действительно построена вокруг агентной постановки.

In [ ]:
important_dirs = ['agent', 'base', 'data', 'env', 'graphics', 'logs_baseline', 'scripts', 'training', 'verifier']
summary = []
for d in important_dirs:
    p = PROJECT_ROOT / d
    summary.append({
        'path': d,
        'exists': p.exists(),
        'n_items': len(list(p.iterdir())) if p.exists() and p.is_dir() else None
    })

pd.DataFrame(summary)

## 4. Baseline: что показывает исходная политика

Сначала рассмотрим baseline-метрики. Они нужны как sanity check: если baseline показывает ненулевой успех хотя бы на части задач, значит сама среда не является полностью нерешаемой.

Ниже загружаются json-файлы из `logs_baseline/`.

In [ ]:
baseline_files = sorted(BASELINE_DIR.glob('metrics_baseline_eval_d*.json'))
if not baseline_files:
    raise FileNotFoundError('Не найдены baseline-метрики в logs_baseline/metrics_baseline_eval_d*.json')

rows = []
for path in baseline_files:
    with open(path, 'r', encoding='utf-8') as f:
        row = json.load(f)
    row['source_file'] = path.name
    rows.append(row)

baseline_df = pd.DataFrame(rows)
if 'bucket' in baseline_df.columns:
    baseline_df['bucket_id'] = baseline_df['bucket'].astype(str).str.extract(r'd(\d+)').astype(float)
    baseline_df = baseline_df.sort_values('bucket_id')

baseline_df.round(4)

### Интерпретация baseline

Если baseline имеет ненулевой success rate на части бакетов, это означает:
- среда работает;
- verifier не сломан;
- агент способен иногда решать задачу даже без RL-дообучения;
- сравнение с GRPO имеет смысл.

Именно поэтому baseline здесь важен: он показывает, что проблема не в том, что задача полностью нерешаемая, а в том, как именно обучение меняет политику.

## 5. Графики эксперимента

В проекте уже сохранены готовые картинки в папке `graphics/`. Ниже они вставляются в отчёт. Если какие-то картинки отсутствуют, ячейка просто пропустит их.

In [ ]:
images = [
    ('vllm_training_curves.png', 'Кривые обучения vLLM / RL'),
    ('grpo_eval_main_metrics.png', 'Основные метрики GRPO на eval'),
    ('grpo_eval_runtime_tools.png', 'Runtime и число tool calls'),
    ('grpo_trajectory_analysis.png', 'Анализ траекторий GRPO'),
    ('grpo_vs_baseline_deltas.png', 'Разница между GRPO и baseline'),
]

for fname, title in images:
    path = GRAPHICS_DIR / fname
    if path.exists():
        display(Markdown(f'### {title}'))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f'### {title}
Файл `{fname}` не найден.'))

## 6. Что видно по результатам

По итоговым графикам и метрикам можно сделать несколько ключевых наблюдений.

### 6.1. Baseline не нулевой

Это важный факт. Он показывает, что исходная политика хотя бы иногда доходит до решения. Значит, среда и инструменты в принципе допускают полезное поведение.

### 6.2. После GRPO политика не улучшилась, а деградировала

Судя по полученным графикам, основная проблема состоит не в том, что обучение «ничего не изменило», а в том, что оно **сдвинуло политику в плохую стратегию**. В типичном виде это выглядит так:
- агент делает меньше шагов;
- раньше завершает эпизод;
- реже пользуется инструментами;
- формально избегает некоторых штрафов;
- но при этом чаще не решает задачу.

### 6.3. Возник collapse в короткие траектории

Это, вероятно, главный содержательный вывод работы. Вместо того чтобы учиться длинному и полезному взаимодействию со средой, агент выучивает стратегию минимизации риска: короткий эпизод, мало действий, ранний финальный ответ.

То есть оптимизируется не реальная полезность поведения, а более простой прокси-критерий — **избегание штрафов**.

## 7. Почему так произошло

Наиболее правдоподобная интерпретация результатов — **несбалансированный reward shaping**.

В агентной постановке с инструментами итоговый reward обычно состоит из двух частей:
1. **Outcome reward** — получил ли агент правильный финальный результат.
2. **Shaping penalties / bonuses** — сколько шагов сделал агент, сколько вызовов инструментов использовал, были ли нарушения policy rules, invalid actions и т.д.

Если shaping-компонента слишком сильная, агент начинает оптимизировать не решение задачи, а сокращение своих действий. Тогда у него появляется «рациональная», но плохая стратегия:
- не исследовать среду;
- не тратить шаги на промежуточные запросы;
- не использовать tools там, где это могло бы помочь;
- как можно быстрее завершать эпизод.

Именно это и похоже на observed behavior после GRPO.

## 8. Анализ как исследовательский результат

Хотя с точки зрения прироста качества результат нельзя назвать успешным, с исследовательской точки зрения он полезен.

Эксперимент показал типичную проблему RL для tool-using агентов:

> Если прокси-награда составлена неудачно, модель может научиться избегать штрафов вместо того, чтобы решать задачу.

Это содержательный и защитимый вывод. Он показывает, что:
- среда и verifier действительно различают хорошие и плохие траектории;
- baseline служит рабочей точкой сравнения;
- RL-дообучение в такой постановке чувствительно к дизайну reward;
- для агентов с tools особенно важно не переусилить штрафы за длинные траектории.

## 9. Что можно улучшить

Ниже приведены разумные направления для следующей итерации эксперимента.

1. **Ослабить штраф за шаги и tool calls.**
   Если штраф слишком велик, агенту выгоднее ничего не делать.

2. **Сильнее вознаграждать полезные промежуточные действия.**
   Например, корректные обращения к инструментам, которые реально уточняют состояние среды.

3. **Наказывать слишком ранний финальный ответ.**
   Особенно если до этого агент не получил достаточно информации из среды.

4. **Сделать curriculum мягче.**
   Возможно, политику стоит учить сначала на более простых случаях с меньшей ценой exploration.

5. **Перебалансировать reward между outcome и shaping.**
   Финальный успех должен оставаться доминирующей целью.

6. **Увеличить разнообразие rollout’ов и посмотреть на более длинные траектории.**
   Иногда collapse возникает из-за того, что модель слишком быстро закрепляет короткий паттерн поведения.

## 10. Итоговые выводы

По итогам работы можно сделать следующие выводы.

1. Была реализована и использована корректная постановка с **tool-using агентом**, средой, verifier и reward function.
2. Baseline показывает, что задача не является полностью нерешаемой.
3. Дообучение с помощью GRPO в текущей конфигурации **не улучшило** политику.
4. Наоборот, наблюдается деградация и collapse в короткие траектории.
5. Наиболее вероятная причина — **reward misspecification / reward hacking**, когда агент учится минимизировать penalties вместо реального решения задачи.
6. Поэтому главный результат работы — не рост метрик, а демонстрация типичного failure mode RL для агентов, использующих инструменты.

В таком виде результат остаётся содержательным: он показывает, что для agentic RL важен не только сам алгоритм обучения, но и очень аккуратный дизайн reward function.

## 11. Приложение: компактная таблица baseline-метрик

In [ ]:
cols = [c for c in [
    'bucket', 'success_rate', 'mean_reward', 'mean_steps',
    'mean_tool_calls', 'mean_policy_violations', 'time_seconds'
] if c in baseline_df.columns]

baseline_df[cols].round(4)